# Lab 4 Parte 2 — Solución LSTM SHM (solo docente)

Referencia con hiperparámetros recomendados. Alumnos: `rnn_sensores_estructuras_alumno_ia.ipynb`.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from _verificar import (
    verificar_panorama_rnn, verificar_carga, verificar_orden_temporal,
    verificar_limpieza, verificar_columna, verificar_etiquetas,
    verificar_serie_temporal, verificar_series_condicion, verificar_ventanas,
    verificar_arquitectura_lstm, verificar_entrenamiento_lstm,
    verificar_interpolacion, verificar_extrapolacion, resumen_seccion,
)

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
from _modelo_lstm import LSTMClassifier, StrainLSTM, load_lstm_classifier, load_strain_lstm
from torch.utils.data import DataLoader, TensorDataset

%matplotlib inline
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cpu')
print(f"✅ Entorno listo | device={device}")


## Contexto del dataset (Kaggle SHM)

| Variable | Unidad | En obra significa… |
|----------|--------|-------------------|
| Accel_X, Accel_Y, Accel_Z | m/s² | Vibración en tres ejes |
| Strain | με | Deformación (extensómetro) |
| Temp | °C | Temperatura del sensor |
| **Condition Label** | **0 / 1 / 2** | **Saludable / daño menor / severo** |

Mismo CSV que **Lab 1** y **Lab 3**. Aquí explotamos el **orden temporal** con LSTM.

Detalle: [`data/DATOS.md`](data/DATOS.md).


## Pregunta 1 — Panorama RNN/LSTM en monitoreo estructural

Las **RNN/LSTM** procesan **secuencias** de sensores — la memoria temporal ayuda a detectar patrones de daño en el tiempo.

### 📘 Subpreguntas
- **1.a** ¿Qué aporta LSTM frente a XGBoost (Lab 3) en este dataset?
- **1.b** ¿Qué es una ventana deslizante?
- **1.c** ¿Por qué no barajamos el tiempo al hacer train/val?


#### ✅ Respuesta sugerida

**1.a** LSTM usa el historial reciente de sensores; XGBoost trata cada fila sin memoria explícita.
**1.b** Ventana = últimos W segundos de lecturas como entrada a la red.
**1.c** Barajar rompe la causalidad y filtra información del futuro al pasado (fuga temporal).


In [ ]:
COMPONENTES_RNN = ["ventana temporal", "LSTM", "hidden size", "dropout", "fully connected"]
print("Componentes RNN del laboratorio:")
for c in COMPONENTES_RNN:
    print(f"  · {c}")


In [ ]:
# --- Autoevaluación 1 ---
# Requiere (celda «Aquí coloca tu solución»): `COMPONENTES_RNN`
r = []
try:
    r = verificar_panorama_rnn(COMPONENTES_RNN)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('1 — Panorama RNN', r)


## Pregunta 2 — Carga y orden temporal

### 📘 Subpreguntas
- **2.a** ¿Cuántas features de sensor vs 1 etiqueta?
- **2.b** ¿Por qué ordenamos por `Timestamp`?


#### ✅ Reflexión 2

5 sensores + Timestamp + label. Orden cronológico es obligatorio.


In [ ]:
# --- PRE-ESCRITO: carga y orden cronológico ---
RUTA_DATOS = Path("data/building_health_monitoring_dataset.csv")
if not RUTA_DATOS.is_file():
    raise FileNotFoundError("Ejecuta: python _preparar_datos.py o bash labs/setup.sh")
df = pd.read_csv(RUTA_DATOS)
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values('Timestamp').reset_index(drop=True)
print(f"Archivo: {RUTA_DATOS} | Forma: {df.shape[0]} × {df.shape[1]}")


In [ ]:
FEATURES = [
    "Accel_X (m/s^2)", "Accel_Y (m/s^2)", "Accel_Z (m/s^2)",
    "Strain (με)", "Temp (°C)",
]
N_FILAS_HEAD = 5
print(f"Features: {FEATURES}")
display(df.head(N_FILAS_HEAD))


In [ ]:
# --- Autoevaluación 2 ---
# Requiere (celda «Aquí coloca tu solución»): `FEATURES`, `N_FILAS_HEAD`
r = []
try:
    r = verificar_carga(df, N_FILAS_HEAD, FEATURES)
    r.extend(verificar_orden_temporal(df))
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('2 — Carga', r)


## Pregunta 3 — Calidad de datos y balance

### 📘 Subpreguntas
- **3.a** ¿Cuántas filas se pierden al eliminar nulos?
- **3.b** ¿Por qué `Strain` suele ser crítico en SHM?


#### ✅ Reflexión 3

96 filas con nulos; Strain domina en daño.


In [ ]:
# --- PRE-ESCRITO: limpieza y balance ---
n_antes = len(df)
df_limpio = df.dropna(subset=FEATURES).copy()
n_despues = len(df_limpio)
conteo = df_limpio['Condition Label'].value_counts().sort_index().to_dict()
print(f"Tras dropna: {n_antes} → {n_despues}")
fig, ax = plt.subplots(figsize=(6, 4))
pd.Series(conteo).plot(kind='bar', ax=ax, color=['#2ecc71', '#f39c12', '#e74c3c'])
ax.set_title('Distribución de Condition Label')
plt.tight_layout()
plt.show()


In [ ]:
COLUMNA_REVISAR = "Strain (με)"
stats_col = df[COLUMNA_REVISAR].describe()
print(f"Estadísticas «{COLUMNA_REVISAR}» (crudo):")
display(stats_col)


In [ ]:
# --- Autoevaluación 3 ---
# Requiere (celda «Aquí coloca tu solución»): `COLUMNA_REVISAR`
r = []
try:
    r = verificar_limpieza(df_limpio, n_antes, n_despues)
    r.extend(verificar_columna(df, COLUMNA_REVISAR))
    r.extend(verificar_etiquetas(conteo))
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('3 — Calidad', r)


## Pregunta 4 — Serie temporal global (EDA)

Antes de entrenar, **visualiza** cómo evolucionan los sensores en el tiempo.

### 📘 Subpreguntas
- **4.a** ¿Ves cambios de amplitud en Strain antes de cambios de etiqueta?
- **4.b** ¿Hay ruido o tendencias lentas (temperatura)?


#### ✅ Reflexión 4

Visualizar Strain revela picos antes de cambios de etiqueta.


In [ ]:
SENSOR_EDA = "Strain (με)"
N_PUNTOS_PLOT = 300
serie = df_limpio[SENSOR_EDA].iloc[:N_PUNTOS_PLOT]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(serie.values, color='#2980b9', linewidth=0.8)
ax.set_title(f'Serie temporal — {SENSOR_EDA}')
ax.set_xlabel('Índice temporal (1 Hz)'); ax.set_ylabel(SENSOR_EDA)
plt.tight_layout()
plt.show()


In [ ]:
# --- Autoevaluación 4 ---
# Requiere (celda «Aquí coloca tu solución»): `SENSOR_EDA`, `N_PUNTOS_PLOT`
r = []
try:
    r = verificar_serie_temporal(SENSOR_EDA, N_PUNTOS_PLOT, df_limpio)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('4 — Serie temporal', r)


## Pregunta 5 — Series por condición estructural

### 📘 Subpreguntas
- **5.a** ¿La clase 2 (severo) muestra Strain distinto?
- **5.b** ¿Qué sensor complementarías en obra?


#### ✅ Reflexión 5

Clase 2 suele mostrar Strain más variable.


In [ ]:
SENSORES_COMPARAR = ["Strain (με)", "Temp (°C)"]
fig, axes = plt.subplots(1, len(SENSORES_COMPARAR), figsize=(10, 4))
if len(SENSORES_COMPARAR) == 1:
    axes = [axes]
colores = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}
for ax, sensor in zip(axes, SENSORES_COMPARAR):
    for clase in [0, 1, 2]:
        sub = df_limpio[df_limpio['Condition Label'] == clase][sensor].iloc[:100]
        ax.plot(sub.values, label=f'Clase {clase}', color=colores[clase], alpha=0.8)
    ax.set_title(sensor); ax.legend(fontsize=8)
plt.suptitle('Sensores por Condition Label (primeros 100 puntos/clase)')
plt.tight_layout()
plt.show()


In [ ]:
# --- Autoevaluación 5 ---
# Requiere (celda «Aquí coloca tu solución»): `SENSORES_COMPARAR`
r = []
try:
    r = verificar_series_condicion(SENSORES_COMPARAR)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('5 — Series por condición', r)


## Pregunta 6 — Ventanas deslizantes y split temporal

### 📘 Subpreguntas
- **6.a** ¿Qué predice el label de la ventana (último instante)?
- **6.b** ¿Por qué train son los primeros 80 % de ventanas?


#### ✅ Reflexión 6

Label = estado en el último instante de la ventana.


In [ ]:
# --- PRE-ESCRITO: ventanas + split temporal (sin shuffle en val) ---
def make_sequence_loaders(df_seq, features, window_size, batch_size):
    X_raw = df_seq[features].values
    y_raw = df_seq['Condition Label'].values.astype(int)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)
    seqs, labels = [], []
    for i in range(window_size, len(X_scaled)):
        seqs.append(X_scaled[i - window_size : i])
        labels.append(y_raw[i])
    X_arr = np.array(seqs, dtype=np.float32)
    y_arr = np.array(labels, dtype=np.int64)
    cut = int(0.8 * len(X_arr))
    X_train, X_val = X_arr[:cut], X_arr[cut:]
    y_train, y_val = y_arr[:cut], y_arr[cut:]
    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, len(X_train), len(X_val)

print("✅ make_sequence_loaders listo (split 80/20 temporal).")


In [ ]:
WINDOW_SIZE = 30
BATCH_SIZE = 32
train_loader, val_loader, n_train, n_val = make_sequence_loaders(
    df_limpio, FEATURES, WINDOW_SIZE, BATCH_SIZE)
xb, yb = next(iter(train_loader))
print(f"Ventanas train={n_train} val={n_val} | batch X={tuple(xb.shape)} y={tuple(yb.shape)}")


In [ ]:
# --- Autoevaluación 6 ---
# Requiere (celda «Aquí coloca tu solución»): `WINDOW_SIZE`, `BATCH_SIZE`, `train_loader`, `val_loader`
r = []
try:
    r = verificar_ventanas(WINDOW_SIZE, BATCH_SIZE, train_loader, val_loader, n_train, n_val)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('6 — Ventanas', r)


## Pregunta 7 — Arquitectura LSTM

### 📘 Subpreguntas
- **7.a** ¿Qué hace `batch_first=True`?
- **7.b** ¿Por qué tomamos el último estado oculto?


#### ✅ Reflexión 7

Último hidden resume la ventana reciente.


In [ ]:
HIDDEN_SIZE = 64
N_LAYERS = 1
DROPOUT = 0.2

modelo = LSTMClassifier(
    input_size=len(FEATURES), hidden_size=HIDDEN_SIZE,
    n_layers=N_LAYERS, dropout=DROPOUT,
).to(device)
print(modelo)


In [ ]:
# --- Autoevaluación 7 ---
# Requiere (celda «Aquí coloca tu solución»): `modelo`, `HIDDEN_SIZE`, `N_LAYERS`, `DROPOUT`
r = []
try:
    r = verificar_arquitectura_lstm(modelo, HIDDEN_SIZE, N_LAYERS, DROPOUT)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('7 — LSTM', r)


## Pregunta 8 — Entrenamiento y métricas

### 📘 Subpreguntas
- **8.a** ¿Val accuracy sube con las épocas?
- **8.b** ¿Qué clase es más difícil de detectar?


#### ✅ Reflexión 8

Accuracy val ~0.7–0.9 según seed; revisar recall clase 2.


In [ ]:
# --- PRE-ESCRITO: entrenamiento LSTM ---
def train_one_epoch(model, loader, criterion, optimizer, dev):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(dev), yb.to(dev)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += yb.size(0)
    return loss_sum / total, correct / total

def eval_epoch(model, loader, criterion, dev):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(dev), yb.to(dev)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss_sum += loss.item() * xb.size(0)
            correct += (logits.argmax(1) == yb).sum().item()
            total += yb.size(0)
    return loss_sum / total, correct / total

print("✅ Funciones train_one_epoch / eval_epoch listas.")


## Pregunta 7b — Modelo docente LSTM (referencia)

`data/lstm_classifier_best.pt` es el clasificador entrenado más tiempo por el docente. Lo cargamos para comparar con tu LSTM corto.

### 📘 Subpreguntas
- **7b.a** ¿Qué `hidden_size` y épocas usó el docente?
- **7b.b** ¿Por qué entrenamos poco en CPU en clase?


In [ ]:
# --- PRE-ESCRITO: clasificador docente ---
modelo_docente, meta_docente = load_lstm_classifier(device=device)
hp_docente = meta_docente.get('classifier', {}).get('hyperparameters', {})
N_EPOCHS_DOCENTE = hp_docente.get('epochs', 10)
acc_docente = meta_docente.get('classifier', {}).get('val_accuracy', hp_docente.get('val_accuracy', 0.0))
_, acc_docente_live = eval_epoch(modelo_docente, val_loader, nn.CrossEntropyLoss(), device)
acc_docente = float(acc_docente_live)
print(f"Modelo docente LSTM | acc val = {acc_docente:.3f}")
print(f"Hiperparámetros docente: {hp_docente}")


In [ ]:
N_EPOCHS = 3
LEARNING_RATE = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(modelo.parameters(), lr=LEARNING_RATE)
history = {k: [] for k in ['train_loss', 'val_loss', 'train_acc', 'val_acc']}
for epoch in range(N_EPOCHS):
    tl, ta = train_one_epoch(modelo, train_loader, criterion, optimizer, device)
    vl, va = eval_epoch(modelo, val_loader, criterion, device)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['train_acc'].append(ta)
    history['val_acc'].append(va)
    print(f"Época {epoch+1}/{N_EPOCHS} | loss {tl:.3f}/{vl:.3f} | acc {ta:.3f}/{va:.3f}")
acc_val = history['val_acc'][-1]
print(f"✅ Entrenamiento completado | acc_val={acc_val:.3f}")


In [ ]:
# --- Autoevaluación 8 ---
# Requiere (celda «Aquí coloca tu solución»): `history`, `N_EPOCHS`, `LEARNING_RATE`, `acc_val`
r = []
try:
    r = verificar_entrenamiento_lstm(history, N_EPOCHS, LEARNING_RATE, acc_val)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('8 — Entrenamiento', r)


## Pregunta 9 — Comparación: tu LSTM vs docente

### 📘 Subpreguntas
- **9.a** ¿Cuánto gana el docente en accuracy?
- **9.b** ¿Más `hidden_size` o más épocas explican la brecha?


In [ ]:
acc_alumno = acc_val
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(
    [f'Tu LSTM\n({N_EPOCHS} ép.)', f'Docente\n({N_EPOCHS_DOCENTE} ép.)'],
    [acc_alumno, acc_docente],
    color=['#9b59b6', '#27ae60'],
)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy validación')
ax.set_title('Comparación clasificador alumno vs docente')
for i, v in enumerate([acc_alumno, acc_docente]):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.show()


In [ ]:
# --- Autoevaluación 9 (comparación) ---
r = []
try:
    r = verificar_comparacion_docente_lstm(acc_alumno, acc_docente, N_EPOCHS, N_EPOCHS_DOCENTE)
except NameError as err:
    print(f"❌ Falta: {err}")
    r = [False]
resumen_seccion('9 — Comparación docente', r)


In [ ]:
# --- PRE-ESCRITO: regresor Strain docente + helpers interp/extrap ---
strain_raw = df_limpio['Strain (με)'].values.astype(np.float32)
strain_norm = (strain_raw - strain_raw.mean()) / (strain_raw.std() + 1e-8)
SEGMENTO_LEN = 120
W_EXTRAP = 80

modelo_reg, meta_strain = load_strain_lstm(device=device)
hp_strain = meta_strain.get('strain_regressor', {}).get('hyperparameters', {})
mae_interp_docente = meta_strain.get('strain_regressor', {}).get('mae_interp', 0.71)
mae_extrap_docente = meta_strain.get('strain_regressor', {}).get('mae_extrap', 0.90)
print(f"Regresor Strain docente | MAE ref interp={mae_interp_docente:.3f} extrap={mae_extrap_docente:.3f}")
print(f"Hiperparámetros: {hp_strain}")
print("Compara tus MAE con estos valores de referencia (entrenado más tiempo).")

def evaluar_interpolacion(model, serie, gap_inicio, gap_fin, seg_len=SEGMENTO_LEN):
    seg = serie[200 : 200 + seg_len].copy()
    entrada = seg.copy()
    entrada[gap_inicio:gap_fin] = 0.0
    x = torch.tensor(entrada[:, None], dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(x).squeeze().cpu().numpy()
    real = seg[gap_inicio:gap_fin]
    pred_gap = pred[gap_inicio:gap_fin]
    mae = float(np.mean(np.abs(real - pred_gap)))
    return mae, real, pred_gap

def evaluar_extrapolacion(model, serie, w, horizonte, start=300):
    full_len = w + horizonte
    hist = serie[start : start + w]
    real_fut = serie[start + w : start + w + horizonte]
    entrada = np.zeros(full_len, dtype=np.float32)
    entrada[:w] = hist
    x = torch.tensor(entrada[:, None], dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(x).squeeze().cpu().numpy()
    pred_fut = pred[w : w + horizonte]
    mae = float(np.mean(np.abs(real_fut - pred_fut)))
    return mae, hist, real_fut, pred_fut

print("✅ modelo_reg cargado desde checkpoint | helpers listos.")


## Pregunta 9 — Test de interpolación (Strain)

Rellenar un **hueco interior** en la serie (valores enmascarados) y comparar con la señal real.

### 📘 Subpreguntas
- **9.a** ¿El modelo captura la tendencia local del hueco?
- **9.b** ¿Interpolar es más fácil que extrapolar? (compararás en §10)


In [ ]:
GAP_INICIO = 40
GAP_FIN = 60
mae_interp, y_real, y_pred = evaluar_interpolacion(modelo_reg, strain_norm, GAP_INICIO, GAP_FIN)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(GAP_INICIO, GAP_FIN), y_real, 'o-', label='Real', color='#2c3e50')
ax.plot(range(GAP_INICIO, GAP_FIN), y_pred, 's--', label='Predicho', color='#e74c3c')
ax.set_title('Interpolación — hueco en Strain (normalizado)')
ax.legend(); plt.tight_layout(); plt.show()
print(f"MAE interpolación: {mae_interp:.4f}")


In [ ]:
# --- Autoevaluación 9 ---
# Requiere (celda «Aquí coloca tu solución»): `mae_interp`, `GAP_INICIO`, `GAP_FIN`
r = []
try:
    r = verificar_interpolacion(mae_interp, GAP_INICIO, GAP_FIN)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('9 — Interpolación', r)


## Pregunta 10 — Test de extrapolación (Strain)

Predecir pasos **futuros** solo con el histórico — más exigente que rellenar un hueco.

### 📘 Subpreguntas
- **10.a** ¿El error crece con el horizonte?
- **10.b** ¿Usarías esto para alertas sin validación humana?


In [ ]:
HORIZONTE_EXTRAP = 15
mae_extrap, hist, real_fut, pred_fut = evaluar_extrapolacion(
    modelo_reg, strain_norm, W_EXTRAP, HORIZONTE_EXTRAP)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(len(hist)), hist, label='Histórico', color='#3498db')
off = len(hist)
ax.plot(range(off, off + len(real_fut)), real_fut, 'o-', label='Real futuro', color='#2c3e50')
ax.plot(range(off, off + len(pred_fut)), pred_fut, 's--', label='Predicho', color='#e74c3c')
ax.axvline(off - 0.5, color='gray', linestyle=':', label='Inicio forecast')
ax.set_title('Extrapolación — forecast Strain (normalizado)')
ax.legend(); plt.tight_layout(); plt.show()
print(f"MAE extrapolación: {mae_extrap:.4f}")


In [ ]:
# --- Autoevaluación 10 ---
# Requiere (celda «Aquí coloca tu solución»): `mae_extrap`, `HORIZONTE_EXTRAP`
r = []
try:
    r = verificar_extrapolacion(mae_extrap, HORIZONTE_EXTRAP, mae_interp)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('10 — Extrapolación', r)


## Pregunta 11 — Reflexión ingeniería

### 📘 Subpreguntas
- **11.a** ¿Cuándo elegirías LSTM vs XGBoost en un edificio instrumentado?
- **11.b** ¿Por qué el split temporal es obligatorio?
- **11.c** ¿Confiarías en extrapolación para cerrar un puente?


#### ✅ Respuesta sugerida

- **LSTM** si hay patrones en el tiempo; **XGBoost** si cada lectura es casi independiente y quieres xAI tabular (Lab 3).
- Split temporal evita que el modelo vea el futuro en entrenamiento.
- Extrapolación solo como apoyo; alertas requieren inspección y redundancia de sensores.


## Cierre docente

EDA temporal + LSTM + tests interp/extrap anclan el deep learning a sensores reales.
